# 🌳 Day 03a: Composite & Flyweight Patterns

---

## 🎯 What You'll Master Today
- **Composite**: treat individual objects and groups uniformly (tree structures)
- **Flyweight**: share common state to save memory
- Real-world: file systems, UI component trees, game particles, text rendering

---

## 🎭 The Analogies

**Composite** = Company org chart  
A department contains teams. A team contains people. You can ask "total salary" at any level — department, team, or person — same interface.

**Flyweight** = Fonts in a document  
A 1000-page book uses letter 'e' thousands of times. Instead of 10,000 separate 'e' objects, store ONE 'e' glyph and reuse it everywhere with different positions.

---

In [ ]:
# ============================================
# 1. COMPOSITE — File System
# ============================================
from abc import ABC, abstractmethod

class FileSystemItem(ABC):
    """Component — common interface."""
    @abstractmethod
    def get_size(self) -> int: pass
    
    @abstractmethod
    def display(self, indent: int = 0) -> str: pass

class File(FileSystemItem):
    """Leaf — no children."""
    def __init__(self, name: str, size: int):
        self.name = name
        self.size = size
    
    def get_size(self) -> int:
        return self.size
    
    def display(self, indent: int = 0) -> str:
        return f"{'  ' * indent}📄 {self.name} ({self.size} KB)"

class Folder(FileSystemItem):
    """Composite — has children (files or folders)."""
    def __init__(self, name: str):
        self.name = name
        self._children: list[FileSystemItem] = []
    
    def add(self, item: FileSystemItem) -> 'Folder':
        self._children.append(item)
        return self  # fluent interface
    
    def get_size(self) -> int:
        return sum(child.get_size() for child in self._children)
    
    def display(self, indent: int = 0) -> str:
        lines = [f"{'  ' * indent}📁 {self.name}/ ({self.get_size()} KB)"]
        for child in self._children:
            lines.append(child.display(indent + 1))
        return "\n".join(lines)


# Build tree
root = Folder("project")
src = Folder("src")
src.add(File("main.py", 15)).add(File("utils.py", 8))

tests = Folder("tests")
tests.add(File("test_main.py", 12))

root.add(src).add(tests).add(File("README.md", 3))

# Same interface at every level
print(root.display())
print(f"\nTotal project size: {root.get_size()} KB")
print(f"src/ size: {src.get_size()} KB")

In [ ]:
# ============================================
# 2. FLYWEIGHT — Game Particle System
# ============================================
import sys

# Without Flyweight — each particle stores everything
class HeavyParticle:
    def __init__(self, x, y, sprite_data, color, texture):
        self.x = x
        self.y = y
        self.sprite_data = sprite_data  # 1 KB shared data
        self.color = color
        self.texture = texture


# With Flyweight — shared (intrinsic) state separated
class ParticleType:
    """Flyweight — shared state (intrinsic)."""
    def __init__(self, sprite_data: str, color: str, texture: str):
        self.sprite_data = sprite_data
        self.color = color
        self.texture = texture

class Particle:
    """Context — unique state (extrinsic) + reference to flyweight."""
    def __init__(self, x: float, y: float, particle_type: ParticleType):
        self.x = x  # unique per particle
        self.y = y  # unique per particle
        self.type = particle_type  # shared reference!
    
    def render(self) -> str:
        return f"Particle at ({self.x}, {self.y}) | {self.type.color}"


class ParticleFactory:
    """Flyweight Factory — ensures sharing."""
    _types: dict[str, ParticleType] = {}
    
    @classmethod
    def get_type(cls, sprite: str, color: str, texture: str) -> ParticleType:
        key = f"{sprite}_{color}_{texture}"
        if key not in cls._types:
            cls._types[key] = ParticleType(sprite, color, texture)
        return cls._types[key]


# Create 1000 particles — only 3 ParticleType objects!
import random
particles = []
configs = [("fire_sprite", "red", "glow"), 
           ("smoke_sprite", "gray", "soft"),
           ("spark_sprite", "yellow", "bright")]

for _ in range(1000):
    sprite, color, texture = random.choice(configs)
    ptype = ParticleFactory.get_type(sprite, color, texture)
    particles.append(Particle(random.uniform(0, 800), random.uniform(0, 600), ptype))

print(f"Total particles: {len(particles)}")
print(f"Unique ParticleType objects: {len(ParticleFactory._types)}")
print(f"\nFirst 3: {[p.render() for p in particles[:3]]}")

```
╔═════════════════════════════════════════════════════════╗
║              FLYWEIGHT MEMORY SAVINGS                   ║
╠═════════════════════════════════════════════════════════╣
║                                                         ║
║  Without Flyweight:                                     ║
║    1000 particles × (x + y + sprite + color + texture)  ║
║    = 1000 × full object = HEAVY                         ║
║                                                         ║
║  With Flyweight:                                        ║
║    3 ParticleType objects (shared)                       ║
║    + 1000 lightweight Particle (x, y, ref)              ║
║    = MUCH LIGHTER                                       ║
║                                                         ║
║  Intrinsic state = shared (ParticleType)                ║
║  Extrinsic state = unique (x, y position)               ║
║                                                         ║
╚═════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is the Composite pattern? | Treats leaf and container uniformly. Same interface for single object and group |
| 2 | Where do you see Composite in real systems? | File systems, DOM trees, UI component hierarchies, org charts |
| 3 | What is intrinsic vs extrinsic state? | Intrinsic = shared, immutable (flyweight). Extrinsic = unique, varies per context |
| 4 | Python Flyweight examples? | String interning (`sys.intern`), integer caching (-5 to 256), `__slots__` |
| 5 | Flyweight vs Singleton? | Singleton: one instance total. Flyweight: one instance per unique state. Multiple flyweights can coexist |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Composite = tree structure with uniform interface     │
## │  • Leaf (File) and Composite (Folder) share interface    │
## │  • Flyweight = share intrinsic state, save memory        │
## │  • Factory ensures flyweight sharing                     │
## │  • Python already uses Flyweight: int cache, str intern  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Observer & Strategy** — event-driven systems and swappable algorithms